# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

We'll list out all defined record sets, their fields, and sample a few records for exploratory review. 

In [ ]:
# List all record sets by @id and fields by @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}\n  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, column: {field.column.id if hasattr(field, 'column') else '<none>'})")
    print("")

You can access each record in a record set by the `@id` of the record set, as shown here for the first record set:

In [ ]:
# Preview records for the first record set
if record_sets:
    rs_id = record_sets[0].id
    print(f"Previewing a few records from record set '@id': {rs_id}")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        print(record)
        if i >= 2:  # Show only first 3 records
            break
else:
    print("No record sets defined in the dataset.")

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Reference the record set and field `@id`s as previously shown.

In [ ]:
# Extract all records from each record set into a pandas DataFrame
dataframes = dict()

for rs in record_sets:
    rs_id = rs.id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"DataFrame for record set '@id': {rs_id} with columns: {df.columns.tolist()}")

# Preview the first DataFrame (if available)
if record_sets:
    main_rs_id = record_sets[0].id
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping data.

We'll select a numeric field (such as protein abundance or molecular weight if present) and demonstrate filtering and normalization using its `@id`.

In [ ]:
# Identify a main numeric field (molecular weight or abundance) by @id
main_rs = record_sets[0]  # use first record set

# Attempt to locate typical numeric field
numeric_field_id = None
for field in main_rs.fields:
    if any(sub in field.name.lower() for sub in ["abundance", "mw", "weight", "count", "peptide"]):
        numeric_field_id = field.id
        break

if numeric_field_id is None:
    # Just use the first field with type number or float if available
    for field in main_rs.fields:
        if getattr(field, "data_type", "").lower() in ["number", "float", "integer"]:
            numeric_field_id = field.id
            break

if numeric_field_id is not None and numeric_field_id in dataframes[main_rs.id].columns:
    df = dataframes[main_rs.id]
    # Ensure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = df[numeric_field_id].quantile(0.75)  # upper quartile as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to find a suitable grouping field, e.g., if 'sample', 'type', or similar appears
    group_field_id = None
    for field in main_rs.fields:
        if any(sub in field.name.lower() for sub in ["sample", "type", "group", "condition"]):
            gf_id = field.id
            if gf_id in df.columns:
                group_field_id = gf_id
                break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id}, mean of {numeric_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA in record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of the selected numeric field, if present, and show a boxplot grouped by one of the categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs.id in dataframes and numeric_field_id is not None and numeric_field_id in dataframes[main_rs.id].columns:
    df = dataframes[main_rs.id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for plotting.")

## 6. Conclusion
In this notebook, you explored the FAIR² dataset described by a Croissant schema using the `mlcroissant` library. You examined its schema (using `@id` references), extracted and processed protein records with pandas, and performed basic EDA and visualization on fields such as abundance or molecular weight. This workflow can be extended for more complex biological or statistical analysis as needed.